# 掩码自注意力与多头注意力：一步一步看矩阵

本 Notebook 只解决两个问题：

1. Causal Mask 到底把注意力矩阵的哪些位置变成 0？
2. Multi-Head Attention 怎样把 `[B,S,H]` 变成 `[B,N,S,D]`，又怎样合回来？

PyTorch 操作的含义直接写在代码注释旁，不单独拆成语法课。


## 0. 环境准备

需要 PyTorch；热力图还需要 Matplotlib。若缺少依赖，取消下一格第一行注释。


In [ ]:
# %pip install torch matplotlib

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(7)
print('PyTorch:', torch.__version__)


## 1. 先只理解 Mask，不考虑多头

有 4 个输入位置：`BOS / 我 / 喜欢 / 苹果`。注意力分数矩阵的**行是 Query，列是 Key**。

- 第 0 行只能看第 0 列；
- 第 1 行能看第 0～1 列；
- 第 2 行能看第 0～2 列；
- 第 3 行能看第 0～3 列。


In [ ]:
tokens = ['BOS', '我', '喜欢', '苹果']
S = len(tokens)

# 假设 QK^T / sqrt(D) 已经计算完。
# Shape=[S,S]；行是 Query 位置，列是 Key 位置。
scores = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [1.0, 2.0, 3.0, 4.0],
    [1.0, 2.0, 3.0, 4.0],
    [1.0, 2.0, 3.0, 4.0],
])

# 创建全部为 -inf 的 [S,S] 矩阵。
mask = torch.full((S, S), float('-inf'))

# 只保留主对角线上方的 -inf；对角线和下方变成 0。
# 因此未来位置被遮住，自己和过去仍可见。
mask = torch.triu(mask, diagonal=1)

# 有限分数加 -inf 仍是 -inf。
masked_scores = scores + mask

# 沿每一行的 Key 位置做 Softmax。
# 被遮位置的 exp(-inf)=0，因此概率变成 0。
weights = F.softmax(masked_scores, dim=-1)

print('原始分数:\n', scores)
print('\n因果 Mask:\n', mask)
print('\nSoftmax 后权重:\n', weights)
print('\n每行之和:', weights.sum(dim=-1))


### 1.1 用热力图检查“未来位置为 0”

左图显示可见范围，右图显示 Softmax 权重。右图主对角线上方应该全部为 0。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# isfinite(mask) 为 True 的位置可以被看到。
axes[0].imshow(torch.isfinite(mask).float(), cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Causal visibility (1=visible)')

axes[1].imshow(weights, cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Attention weights after mask')

for ax in axes:
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    ax.set_xticks(range(S))
    ax.set_yticks(range(S))

plt.tight_layout()
plt.show()


### 1.2 为什么有 Mask 仍然可以并行？

`scores` 的四行一次性算出，Mask 也一次性加到整个矩阵。模型不是先算第 0 行再算第 1 行；它并行算全部位置，只是每一行的可见范围不同。


## 2. 多头不是拆句子，而是拆隐藏维

设 `B=1, S=4, H=8, N=2`，则每头维度 `D=H/N=4`。

$$[1,4,8]\rightarrow[1,4,2,4]\rightarrow[1,2,4,4]$$

两个头都看到完整的 4 个 token；每个头使用每个 token 隐藏向量中的一个学习子空间。


In [ ]:
class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()

        # H 必须能平均拆成 N 个头。
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size             # H，例如 8
        self.num_heads = num_heads                 # N，例如 2
        self.head_dim = hidden_size // num_heads   # D=H/N，例如 4

        # Q/K/V 都来自同一个 x，所以是“自注意力”。
        # 但使用三套不同参数，所以 Q、K、V 的数值并不相同。
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        # 多头合并后再做一次输出投影，混合各头信息。
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x):
        # 输入：B 条序列、每条 S 个 token、每个 token H 维。
        B, S, H = x.shape
        print('0. input [B,S,H]:', tuple(x.shape))

        # 三个投影都只变换最后一维；H->H，所以 Shape 不变。
        q = self.q_proj(x)  # [B,S,H]
        k = self.k_proj(x)  # [B,S,H]
        v = self.v_proj(x)  # [B,S,H]
        print('1. projected Q/K/V [B,S,H]:', tuple(q.shape))

        # 把最后的 H 维重解释成 N 个头、每头 D 维。
        # [B,S,H] -> [B,S,N,D]
        q = q.view(B, S, self.num_heads, self.head_dim)
        k = k.view(B, S, self.num_heads, self.head_dim)
        v = v.view(B, S, self.num_heads, self.head_dim)
        print('2. split heads [B,S,N,D]:', tuple(q.shape))

        # 交换 S 和 N，让每个头拥有一张完整的 [S,D] 矩阵。
        # [B,S,N,D] -> [B,N,S,D]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        print('3. transpose [B,N,S,D]:', tuple(q.shape))

        # 每个头独立计算每个 Query 对每个 Key 的点积。
        # [B,N,S,D] @ [B,N,D,S] -> [B,N,S,S]
        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)
        print('4. QK^T [B,N,S,S]:', tuple(scores.shape))

        # [S,S] 的同一份 Mask 会广播给所有 Batch 和 Head。
        mask = torch.full((S, S), float('-inf'), device=x.device)
        mask = torch.triu(mask, diagonal=1)
        scores = scores + mask

        # 最后一维是 Key 位置；每个 Query 沿全部 Key 做 Softmax。
        # 临时转 FP32 可提高 Softmax 的数值稳定性。
        weights = F.softmax(scores.float(), dim=-1).type_as(q)
        print('5. weights [B,N,S,S]:', tuple(weights.shape))

        # 每个头使用概率对 V 加权求和。
        # [B,N,S,S] @ [B,N,S,D] -> [B,N,S,D]
        context = torch.matmul(weights, v)
        print('6. context [B,N,S,D]:', tuple(context.shape))

        # 把 S 放回 N 前面：[B,N,S,D] -> [B,S,N,D]。
        context = context.transpose(1, 2)

        # transpose 后内存通常不连续；整理后将 N×D 合并回 H。
        context = context.contiguous().view(B, S, H)
        print('7. merge heads [B,S,H]:', tuple(context.shape))

        # 输出投影后仍为 [B,S,H]，所以能与输入做残差相加。
        output = self.out_proj(context)
        print('8. output [B,S,H]:', tuple(output.shape))
        return output, weights


In [ ]:
B, S, H, N = 1, 4, 8, 2
x = torch.randn(B, S, H)

attention = CausalMultiHeadSelfAttention(H, N)
output, head_weights = attention(x)

print('\n每个头各有一张注意力矩阵:', tuple(head_weights.shape))
print('第 0 个头:\n', head_weights[0, 0])
print('第 1 个头:\n', head_weights[0, 1])


### 2.1 比较两个头

两个头使用不同投影参数，因此注意力分布通常不同；但它们使用同一个 Causal Mask，所以主对角线上方都必须为 0。


In [ ]:
fig, axes = plt.subplots(1, N, figsize=(5 * N, 4))
if N == 1:
    axes = [axes]

for head_index, ax in enumerate(axes):
    ax.imshow(head_weights[0, head_index].detach(), cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'Head {head_index}')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    ax.set_xticks(range(S))
    ax.set_yticks(range(S))

plt.tight_layout()
plt.show()


## 3. 避免一个 Shape 陷阱

示例中 `S=4`、`D=4`，所以 Score 和 Context 都显示 `[1,2,4,4]`，但语义完全不同：

- Score：`[B,N,S_q,S_kv]`；
- Context：`[B,N,S,D]`。

把序列长度改成 5，就能直观看出区别。


In [ ]:
B, S, H, N = 1, 5, 8, 2
x_five_tokens = torch.randn(B, S, H)
output_five, weights_five = attention(x_five_tokens)

print('\nScore/weights [B,N,S,S]:', tuple(weights_five.shape))
print('Final output [B,S,H]:', tuple(output_five.shape))


## 4. Decode 时 Score 不一定是方阵

使用 KV Cache 时，当前 Query 通常只有 1 个 token，而 K/V 包含全部历史。此时 `S_q=1`、`S_kv>1`。


In [ ]:
B, N, S_q, S_kv, D = 1, 2, 1, 6, 4

q_decode = torch.randn(B, N, S_q, D)
k_cache = torch.randn(B, N, S_kv, D)
v_cache = torch.randn(B, N, S_kv, D)

# [B,N,1,D] @ [B,N,D,S_kv] -> [B,N,1,S_kv]
decode_scores = torch.matmul(q_decode, k_cache.transpose(-2, -1))
decode_scores = decode_scores / math.sqrt(D)

# 当前 Query 面前只有已经生成的历史，没有未来 Cache。
# 这个最简单的单序列例子不需要 Prefill 那种 S×S 上三角 Mask。
decode_weights = F.softmax(decode_scores.float(), dim=-1).type_as(q_decode)

# [B,N,1,S_kv] @ [B,N,S_kv,D] -> [B,N,1,D]
decode_context = torch.matmul(decode_weights, v_cache)

print('Q:', tuple(q_decode.shape))
print('K/V cache:', tuple(k_cache.shape))
print('Score:', tuple(decode_scores.shape))
print('Context:', tuple(decode_context.shape))


## 5. 动手练习

1. 手画 4×4 Causal Mask，并逐行解释谁能看谁。
2. 删除 Mask，观察两个头的未来位置是否出现非零权重。
3. 把 `S` 改成 6，预测 Score 和 Context Shape 后再运行。
4. 把 `N` 改成 4（保持 `H=8`），观察每头的 `D`。
5. 在 Decode 实验中把 `S_kv` 改成 128。

完成后请独立回答：Mask 为什么在 Softmax 前加入？多头拆的是 token 还是隐藏维？Score 最后一维为什么是 Key？Decode 的 Score 为什么不一定是方阵？
